# Owlpaper small-model capstone
Select a **T4 GPU** runtime and run the code cell. This runs a real preflight, then 5,000 numeric examples per arm, three epochs, seed 3. The Mac worker runs seeds 1 and 2 in a separate data/precision cohort.
Model revision: `70d244cc86ccca08cf5af4e1e306ecf908b1ad5e`. Runtime results are temporary until downloaded; leave the notebook open and save results before runtime expiry. No paid GPU is requested.

In [ ]:
# Owlpaper AI safety capstone: isolated small-model Colab cohort.
# NF4/quantization is NOT used. T4 uses FP16 autocast + GradScaler.
# Mac and Colab are independent dataset/precision cohorts, not same-data seeds.
import base64, io, json, pathlib, subprocess, sys, time, zipfile
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers==4.57.6", "peft==0.18.1", "huggingface-hub==0.36.2", "psutil", "matplotlib"], check=True)
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/2**30,2), flush=True)
BUNDLE = "..."
workspace = pathlib.Path("/content")
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall(workspace)
project = workspace / "owlpaper-small"
run = project / "runs/small-qwen17b/colab"
run.mkdir(parents=True, exist_ok=True)
log = (run / "runner.log").open("a")
command = [sys.executable, "-u", str(project / "scripts/12_small_study.py"), "--root", str(run), "--worker", "colab", "--seeds", "3"]
process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, cwd=project)
print("Started:", process.pid, "Run:", run, flush=True)
# No Drive: snapshot resumable state (incl. training checkpoints) every 20 min
# so it can be downloaded from the Files pane if the free runtime is about to end.
snapshot = pathlib.Path("/content/owlpaper-small-colab-snapshot.zip")
def write_snapshot():
    temp = snapshot.with_suffix(".tmp")
    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED) as output:
        for file in run.rglob("*"):
            if file.is_file() and "epoch-" not in str(file.relative_to(run)):
                output.write(file, str(file.relative_to(project)))
    temp.replace(snapshot)
    print(time.strftime("%H:%M:%S"), "snapshot", snapshot, round(snapshot.stat().st_size/2**20,1), "MiB", flush=True)
last_snapshot = time.time()
last_status = None
while process.poll() is None:
    if time.time() - last_snapshot > 1200:
        write_snapshot()
        last_snapshot = time.time()
    status_file = run / "status.json"
    if status_file.exists():
        status = json.loads(status_file.read_text())
        text = (status.get("state"), status.get("stage"))
        if text != last_status:
            print(time.strftime("%H:%M:%S"), text, flush=True)
            last_status = text
    time.sleep(5)
log.close()
print((run / "runner.log").read_text()[-10000:], flush=True)
archive = pathlib.Path("/content/owlpaper-small-colab-results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as output:
    for file in run.rglob("*"):
        if file.is_file() and "epoch-" not in str(file.relative_to(run)) and file.name != "training-state.pt":
            output.write(file, str(file.relative_to(project)))
print("Results archive:", archive, flush=True)
assert process.returncode == 0, "Worker stopped: inspect log/status above; partial results are saved in the archive."
from google.colab import files
files.download(str(archive))
